<a href="https://colab.research.google.com/github/DarioCorona/personal/blob/root/Evaluacion_Modulo5_Machine_Learning_Mantenimiento_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación integradora - Módulo 5

## Machine Learning aplicado al mantenimiento predictivo

**Ingenium - Ciencia de Datos aplicada al Mantenimiento Predictivo**

Esta evaluación integra formulación supervisada, clasificación, regresión, evaluación, optimización, interpretación y persistencia de modelos. Trabajarás con el conjunto real **Condition Monitoring of Hydraulic Systems**, publicado por UCI.

> Completa únicamente las celdas marcadas con **RESPUESTA DEL ESTUDIANTE**. No modifiques la preparación de datos ni el calificador.

## Reglas de trabajo

- Ejecuta las celdas en orden.
- Conserva los nombres de variables solicitados.
- No agregues la variable objetivo, sus copias ni etiquetas del perfil dentro de `X`.
- Utiliza `random_state=42` cuando se indique.
- La prueba no debe participar en la selección de hiperparámetros.
- Las métricas deben calcularse sobre los conjuntos de prueba.
- Guarda el notebook con las salidas y la tabla final de calificación visibles.
- El puntaje máximo es **20 puntos**.

La calificación automática comprueba estructura y resultados. El profesor puede revisar además claridad, coherencia técnica e integridad académica.

## 0. Datos del estudiante

Completa la siguiente celda antes de comenzar.

In [ ]:
# RESPUESTA DEL ESTUDIANTE
NOMBRE_COMPLETO = ""
GRUPO = ""
CORREO = ""


## 1. Preparación del entorno y datos

La celda siguiente descarga automáticamente el ZIP oficial de UCI. El archivo pesa aproximadamente 73 MB. Cada fila final representa un ciclo de trabajo del banco hidráulico y cada característica es el promedio de un sensor físico durante ese ciclo.

**No modifiques esta celda.**

In [ ]:
# CELDA DE PREPARACIÓN - NO MODIFICAR
from pathlib import Path
from zipfile import ZipFile
import gc
import os
import urllib.request

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
DATA_URLS = [
    "https://zenodo.org/records/1323611/files/data.zip?download=1",
    "https://archive.ics.uci.edu/static/public/447/condition%2Bmonitoring%2Bof%2Bhydraulic%2Bsystems.zip",
]
ZIP_PATH = Path("condition_monitoring_hydraulic_systems.zip")

def zip_valido(ruta):
    if not ruta.exists():
        return False
    try:
        with ZipFile(ruta) as archivo:
            return any(Path(nombre).name.lower() == "profile.txt" for nombre in archivo.namelist())
    except Exception:
        return False

if not zip_valido(ZIP_PATH):
    ZIP_PATH.unlink(missing_ok=True)
    errores_descarga = []
    for url in DATA_URLS:
        temporal = ZIP_PATH.with_suffix(".partial")
        temporal.unlink(missing_ok=True)
        try:
            print("Descargando el conjunto hidráulico real...")
            urllib.request.urlretrieve(url, temporal)
            if not zip_valido(temporal):
                raise RuntimeError("La descarga no produjo un ZIP válido.")
            temporal.replace(ZIP_PATH)
            break
        except Exception as error:
            errores_descarga.append(f"{url}: {error}")
            temporal.unlink(missing_ok=True)
    else:
        raise RuntimeError("No fue posible descargar el conjunto de datos. " + " | ".join(errores_descarga))

SENSORES_FISICOS = [
    "PS1", "PS2", "PS3", "PS4", "PS5", "PS6",
    "EPS1", "FS1", "FS2", "TS1", "TS2", "TS3", "TS4", "VS1",
]

def buscar_miembro(zip_file, nombre_archivo):
    coincidencias = [
        nombre for nombre in zip_file.namelist()
        if Path(nombre).name.lower() == nombre_archivo.lower()
    ]
    if not coincidencias:
        raise FileNotFoundError(f"No se encontró {nombre_archivo} dentro del ZIP.")
    return coincidencias[0]

def leer_matriz(zip_file, nombre_archivo):
    miembro = buscar_miembro(zip_file, nombre_archivo)
    return pd.read_csv(
        zip_file.open(miembro),
        sep=r"\s+",
        header=None,
        dtype=np.float32,
    )

with ZipFile(ZIP_PATH) as archivo_zip:
    perfil = leer_matriz(archivo_zip, "profile.txt")
    perfil.columns = [
        "cooler_condition_pct",
        "valve_condition_pct",
        "pump_leakage",
        "accumulator_bar",
        "stable_flag",
    ]

    resumen_sensores = {}
    for sensor in SENSORES_FISICOS:
        matriz = leer_matriz(archivo_zip, f"{sensor}.txt")
        resumen_sensores[f"{sensor}_mean"] = matriz.mean(axis=1).astype("float32")
        del matriz
        gc.collect()
        print(f"Preparado: {sensor}")

    # CE es una señal virtual continua; se usa exclusivamente como objetivo de regresión.
    ce = leer_matriz(archivo_zip, "CE.txt")
    cooling_efficiency_pct = ce.mean(axis=1).astype("float32")
    del ce
    gc.collect()

datos = pd.DataFrame(resumen_sensores)
datos.insert(0, "cycle_id", np.arange(1, len(datos) + 1))
datos = pd.concat([datos, perfil.reset_index(drop=True)], axis=1)
datos["cooling_efficiency_pct"] = cooling_efficiency_pct.reset_index(drop=True)

FEATURE_COLUMNS = [f"{sensor}_mean" for sensor in SENSORES_FISICOS]
TARGET_CLASSIFICATION = "pump_leakage"
TARGET_REGRESSION = "cooling_efficiency_pct"
PROFILE_COLUMNS = [
    "cooler_condition_pct", "valve_condition_pct", "pump_leakage",
    "accumulator_bar", "stable_flag",
]

print()
print("Preparación terminada")
print("Dimensiones:", datos.shape)
print("Características autorizadas:", len(FEATURE_COLUMNS))
print("Clases de fuga interna:", sorted(datos[TARGET_CLASSIFICATION].unique().tolist()))
display(datos.head())


### Diagnóstico de control

Ejecuta la siguiente celda y confirma que existen 2,205 ciclos, 14 características autorizadas y tres clases para `pump_leakage`.

In [ ]:
# CONTROL - NO MODIFICAR
control_datos = pd.DataFrame({
    "indicador": [
        "ciclos", "caracteristicas", "faltantes_en_X",
        "clases_bomba", "objetivo_regresion_min", "objetivo_regresion_max",
    ],
    "valor": [
        len(datos),
        len(FEATURE_COLUMNS),
        int(datos[FEATURE_COLUMNS].isna().sum().sum()),
        int(datos[TARGET_CLASSIFICATION].nunique()),
        float(datos[TARGET_REGRESSION].min()),
        float(datos[TARGET_REGRESSION].max()),
    ],
})
display(control_datos)
display(datos[TARGET_CLASSIFICATION].value_counts().sort_index().rename("ciclos"))


## 2. Formulación de los problemas - 2 puntos

Construye exactamente:

- `X`: solamente las columnas incluidas en `FEATURE_COLUMNS`.
- `y_clasificacion`: la columna `pump_leakage`.
- `y_regresion`: la columna continua `cooling_efficiency_pct`.
- `revision_fuga`: diccionario con las claves `columnas_objetivo_en_X` y `hay_fuga`.

`columnas_objetivo_en_X` debe listar cualquier objetivo o etiqueta del perfil incluida por error en `X`. `hay_fuga` debe ser un booleano.

In [ ]:
# RESPUESTA DEL ESTUDIANTE - TAREA 1
# Escribe tu solución debajo.



## 3. Particiones de entrenamiento y prueba - 2 puntos

### Clasificación

Utiliza `train_test_split` con 25% para prueba, `random_state=42` y estratificación por clase. Conserva estos nombres:

- `X_train_cls`, `X_test_cls`, `y_train_cls`, `y_test_cls`

### Regresión

Reserva cronológicamente el último 25% de los ciclos como prueba, sin barajar. Conserva:

- `X_train_reg`, `X_test_reg`, `y_train_reg`, `y_test_reg`

No ajustes transformaciones antes de realizar las particiones.

In [ ]:
# RESPUESTA DEL ESTUDIANTE - TAREA 2
# Escribe tu solución debajo.



## 4. Clasificación de la fuga interna de la bomba - 5 puntos

1. Entrena `modelo_base_cls` con `DummyClassifier(strategy="most_frequent")`.
2. Crea y entrena tres modelos dentro del diccionario `modelos_clasificacion`:
   - regresión logística con escalamiento dentro de un `Pipeline`;
   - árbol de decisión;
   - Random Forest.
3. Guarda las predicciones en el diccionario `predicciones_clasificacion` usando las mismas claves.
4. Construye `resultados_clasificacion` con las columnas:
   - `modelo`, `accuracy`, `precision_macro`, `recall_macro`, `f1_macro`.
   - Incluye el modelo base y los tres modelos entrenados.
5. Selecciona un modelo en `modelo_clasificacion_seleccionado`.
6. Calcula `matriz_confusion` para el modelo seleccionado.

Todas las métricas deben calcularse sobre `y_test_cls`. Usa `zero_division=0` cuando corresponda.

In [ ]:
# RESPUESTA DEL ESTUDIANTE - TAREA 3
# Escribe tu solución debajo.



## 5. Regresión de la eficiencia de enfriamiento - 4 puntos

1. Entrena `modelo_base_reg` con `DummyRegressor(strategy="mean")`.
2. Crea y entrena tres modelos en `modelos_regresion`:
   - regresión lineal;
   - árbol de regresión;
   - Random Forest de regresión.
3. Guarda las predicciones en `predicciones_regresion`.
4. Construye `resultados_regresion` con:
   - `modelo`, `MAE`, `RMSE`, `R2`.
   - Incluye el modelo base y los tres modelos.
5. Guarda el modelo elegido en `modelo_regresion_seleccionado`.

Calcula las métricas sobre `y_test_reg`. RMSE debe conservar las mismas unidades del objetivo.

In [ ]:
# RESPUESTA DEL ESTUDIANTE - TAREA 4
# Escribe tu solución debajo.



## 6. Optimización sin utilizar la prueba - 4 puntos

Optimiza un Random Forest para clasificación.

1. Crea `pipeline_busqueda` con imputación por mediana y `RandomForestClassifier(random_state=42, n_jobs=-1)`.
2. Define `param_grid` con al menos dos valores para `n_estimators`, `max_depth` y `min_samples_leaf`.
3. Crea `cv_estratificada` con `StratifiedKFold`, al menos 3 folds, barajado y `random_state=42`.
4. Crea y ajusta `busqueda` mediante `GridSearchCV` usando `scoring="f1_macro"` y solamente el conjunto de entrenamiento.
5. Guarda `modelo_optimizado = busqueda.best_estimator_`.
6. Calcula sobre la prueba `metricas_optimizadas`, un diccionario con `accuracy`, `precision_macro`, `recall_macro` y `f1_macro`.

La prueba se utiliza una sola vez después de terminar la búsqueda.

In [ ]:
# RESPUESTA DEL ESTUDIANTE - TAREA 5
# Escribe tu solución debajo.



## 7. Interpretación y persistencia - 2 puntos

1. Obtén la importancia del Random Forest optimizado.
2. Construye `importancias`, un DataFrame ordenado de mayor a menor con las columnas `caracteristica` e `importancia`. Conserva al menos las diez primeras.
3. Guarda el pipeline completo en la ruta `RUTA_MODELO = "modelo_modulo5.joblib"` mediante `joblib.dump`.
4. Comprueba que el archivo puede cargarse y producir predicciones.

La importancia refleja cómo utiliza variables el modelo; no demuestra causalidad física.

In [ ]:
# RESPUESTA DEL ESTUDIANTE - TAREA 6
# Escribe tu solución debajo.



## 8. Conclusión técnica - 1 punto

Crea el diccionario `conclusion_tecnica` con textos propios y estas claves:

- `modelo_clasificacion`
- `metrica_prioritaria`
- `error_mas_costoso`
- `modelo_regresion`
- `interpretacion_mae`
- `limitacion`
- `accion_recomendada`

La conclusión debe relacionar desempeño, tipo de error, tolerancia y decisión de mantenimiento.

In [ ]:
# RESPUESTA DEL ESTUDIANTE - TAREA 7
# Escribe tu solución debajo.



## 9. Calificador automático

Ejecuta esta celda después de completar todas las tareas. Si realizas una corrección, vuelve a ejecutar la tarea modificada y después el calificador.

**No modifiques el calificador.**

In [ ]:
# CALIFICADOR AUTOMÁTICO - NO MODIFICAR
from pathlib import Path as _Path
from sklearn.model_selection import GridSearchCV as _GridSearchCV
from sklearn.dummy import DummyClassifier as _DummyClassifier, DummyRegressor as _DummyRegressor

def _obtener(nombre, valor_por_defecto=None):
    return globals().get(nombre, valor_por_defecto)

def _es_modelo_ajustado(modelo):
    if modelo is None or not hasattr(modelo, "predict"):
        return False
    return any(nombre.endswith("_") for nombre in vars(modelo)) or hasattr(modelo, "steps")

detalle = []

def _registrar(criterio, puntos, maximo, evidencia):
    detalle.append({
        "criterio": criterio,
        "puntos": round(float(puntos), 2),
        "maximo": float(maximo),
        "evidencia": evidencia,
    })

# 1. Formulación - 2 puntos
p = 0
X_ = _obtener("X")
yc_ = _obtener("y_clasificacion")
yr_ = _obtener("y_regresion")
rev_ = _obtener("revision_fuga", {})
if isinstance(X_, pd.DataFrame) and list(X_.columns) == FEATURE_COLUMNS and len(X_) == len(datos):
    p += 0.75
if isinstance(yc_, pd.Series) and yc_.name == TARGET_CLASSIFICATION and len(yc_) == len(datos):
    p += 0.35
if isinstance(yr_, pd.Series) and yr_.name == TARGET_REGRESSION and len(yr_) == len(datos):
    p += 0.35
if isinstance(rev_, dict) and rev_.get("hay_fuga") is False and len(rev_.get("columnas_objetivo_en_X", [])) == 0:
    p += 0.55
_registrar("Formulación y control de fuga", p, 2, "X, objetivos y revisión de fuga")

# 2. Particiones - 2 puntos
p = 0
part_cls = [_obtener(n) for n in ["X_train_cls", "X_test_cls", "y_train_cls", "y_test_cls"]]
if all(v is not None for v in part_cls):
    Xtr, Xte, ytr, yte = part_cls
    if len(Xtr) + len(Xte) == len(datos) and set(Xtr.index).isdisjoint(set(Xte.index)):
        p += 0.5
    if 0.23 <= len(Xte) / len(datos) <= 0.27 and set(ytr.unique()) == set(yc_.unique()) and set(yte.unique()) == set(yc_.unique()):
        p += 0.5
part_reg = [_obtener(n) for n in ["X_train_reg", "X_test_reg", "y_train_reg", "y_test_reg"]]
if all(v is not None for v in part_reg):
    Xtr, Xte, ytr, yte = part_reg
    if len(Xtr) + len(Xte) == len(datos) and 0.23 <= len(Xte) / len(datos) <= 0.27:
        p += 0.5
    if len(Xtr) and len(Xte) and max(Xtr.index) < min(Xte.index) and list(Xtr.index) == sorted(Xtr.index) and list(Xte.index) == sorted(Xte.index):
        p += 0.5
_registrar("Particiones reproducibles", p, 2, "Estratificación y reserva cronológica")

# 3. Clasificación - 5 puntos
p = 0
base_cls = _obtener("modelo_base_cls")
mods_cls = _obtener("modelos_clasificacion", {})
preds_cls = _obtener("predicciones_clasificacion", {})
res_cls = _obtener("resultados_clasificacion")
sel_cls = _obtener("modelo_clasificacion_seleccionado")
mc = _obtener("matriz_confusion")
if isinstance(base_cls, _DummyClassifier) and hasattr(base_cls, "classes_"):
    p += 0.75
if isinstance(mods_cls, dict) and len(mods_cls) >= 3 and all(_es_modelo_ajustado(m) for m in mods_cls.values()):
    nombres = {m.__class__.__name__ for m in mods_cls.values()}
    if any("Pipeline" in n for n in nombres) or any(hasattr(m, "steps") for m in mods_cls.values()):
        p += 0.5
    if any("DecisionTreeClassifier" == n for n in nombres) and any("RandomForestClassifier" == n for n in nombres):
        p += 0.75
if isinstance(preds_cls, dict) and len(preds_cls) >= 3 and all(len(v) == len(_obtener("y_test_cls", [])) for v in preds_cls.values()):
    p += 0.5
cols_cls = {"modelo", "accuracy", "precision_macro", "recall_macro", "f1_macro"}
if isinstance(res_cls, pd.DataFrame) and cols_cls.issubset(res_cls.columns) and len(res_cls) >= 4:
    metricas = res_cls[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].apply(pd.to_numeric, errors="coerce")
    if metricas.notna().all().all() and ((metricas >= 0) & (metricas <= 1)).all().all():
        p += 1.25
if _es_modelo_ajustado(sel_cls):
    p += 0.5
if isinstance(mc, np.ndarray) and mc.shape == (len(np.unique(_obtener("y_test_cls", []))),) * 2 and mc.sum() == len(_obtener("y_test_cls", [])):
    p += 0.75
_registrar("Clasificación y tipos de error", p, 5, "Base, tres modelos, métricas y matriz")

# 4. Regresión - 4 puntos
p = 0
base_reg = _obtener("modelo_base_reg")
mods_reg = _obtener("modelos_regresion", {})
preds_reg = _obtener("predicciones_regresion", {})
res_reg = _obtener("resultados_regresion")
sel_reg = _obtener("modelo_regresion_seleccionado")
if isinstance(base_reg, _DummyRegressor) and hasattr(base_reg, "constant_"):
    p += 0.6
if isinstance(mods_reg, dict) and len(mods_reg) >= 3 and all(_es_modelo_ajustado(m) for m in mods_reg.values()):
    nombres = {m.__class__.__name__ for m in mods_reg.values()}
    if {"LinearRegression", "DecisionTreeRegressor", "RandomForestRegressor"}.issubset(nombres):
        p += 1.0
if isinstance(preds_reg, dict) and len(preds_reg) >= 3 and all(len(v) == len(_obtener("y_test_reg", [])) for v in preds_reg.values()):
    p += 0.4
cols_reg = {"modelo", "MAE", "RMSE", "R2"}
if isinstance(res_reg, pd.DataFrame) and cols_reg.issubset(res_reg.columns) and len(res_reg) >= 4:
    metricas = res_reg[["MAE", "RMSE", "R2"]].apply(pd.to_numeric, errors="coerce")
    if metricas.notna().all().all() and np.isfinite(metricas.to_numpy()).all() and (metricas[["MAE", "RMSE"]] >= 0).all().all():
        p += 1.5
if _es_modelo_ajustado(sel_reg):
    p += 0.5
_registrar("Regresión y magnitud del error", p, 4, "Base, tres regresores y métricas")

# 5. Optimización - 4 puntos
p = 0
pipe = _obtener("pipeline_busqueda")
grid = _obtener("param_grid", {})
cv = _obtener("cv_estratificada")
busq = _obtener("busqueda")
opt = _obtener("modelo_optimizado")
met_opt = _obtener("metricas_optimizadas", {})
if hasattr(pipe, "steps") and any(nombre == "imputer" for nombre, _ in pipe.steps) and any(nombre == "model" for nombre, _ in pipe.steps):
    p += 0.75
if isinstance(grid, dict) and all(any(k.endswith(sufijo) for k in grid) for sufijo in ["n_estimators", "max_depth", "min_samples_leaf"]) and all(len(v) >= 2 for v in grid.values()):
    p += 0.75
if cv is not None and getattr(cv, "n_splits", 0) >= 3:
    p += 0.5
if isinstance(busq, _GridSearchCV) and hasattr(busq, "best_estimator_") and getattr(busq, "scoring", None) == "f1_macro":
    p += 1.25
if opt is not None and opt is getattr(busq, "best_estimator_", None):
    p += 0.25
claves_opt = {"accuracy", "precision_macro", "recall_macro", "f1_macro"}
if isinstance(met_opt, dict) and claves_opt.issubset(met_opt) and all(np.isfinite(float(met_opt[k])) and 0 <= float(met_opt[k]) <= 1 for k in claves_opt):
    p += 0.5
_registrar("Optimización sin abrir la prueba", p, 4, "Pipeline, CV, GridSearch y prueba final")

# 6. Interpretación y persistencia - 2 puntos
p = 0
imp = _obtener("importancias")
ruta = _obtener("RUTA_MODELO")
if isinstance(imp, pd.DataFrame) and {"caracteristica", "importancia"}.issubset(imp.columns) and len(imp) >= 10:
    vals = pd.to_numeric(imp["importancia"], errors="coerce")
    if vals.notna().all() and (vals >= 0).all() and vals.is_monotonic_decreasing:
        p += 1.0
if isinstance(ruta, str) and _Path(ruta).exists():
    try:
        cargado = joblib.load(ruta)
        pred = cargado.predict(_obtener("X_test_cls").iloc[:3])
        if len(pred) == 3:
            p += 1.0
    except Exception:
        pass
_registrar("Interpretación y persistencia", p, 2, "Importancias y pipeline guardado")

# 7. Conclusión - 1 punto
p = 0
conclusion = _obtener("conclusion_tecnica", {})
claves_conclusion = {
    "modelo_clasificacion", "metrica_prioritaria", "error_mas_costoso",
    "modelo_regresion", "interpretacion_mae", "limitacion", "accion_recomendada",
}
if isinstance(conclusion, dict) and claves_conclusion.issubset(conclusion):
    textos = [str(conclusion[k]).strip() for k in claves_conclusion]
    if all(len(t) >= 12 for t in textos):
        p = 1.0
_registrar("Conclusión técnica", p, 1, "Desempeño, riesgo, límite y acción")

tabla_calificacion = pd.DataFrame(detalle)
PUNTAJE_TOTAL = round(tabla_calificacion["puntos"].sum(), 2)
tabla_calificacion.loc[len(tabla_calificacion)] = ["TOTAL", PUNTAJE_TOTAL, 20.0, ""]

display(tabla_calificacion)
print(f"PUNTAJE AUTOMÁTICO: {PUNTAJE_TOTAL:.2f} / 20.00")
if not str(_obtener("NOMBRE_COMPLETO", "")).strip():
    print("ADVERTENCIA: completa NOMBRE_COMPLETO antes de entregar.")


## 10. Entrega

Antes de descargar el notebook:

1. Ejecuta todas las celdas en orden.
2. Verifica que las tablas de resultados y la calificación sean visibles.
3. Guarda el archivo.
4. Descarga una copia `.ipynb`.
5. Renómbrala como `Apellido_Nombre_Evaluacion_Modulo5.ipynb`.
6. Entrega el notebook completo, no solamente capturas de pantalla.

### Fuente del conjunto de datos

Helwig, N., Pignanelli, E. y Schütze, A. (2015). *Condition Monitoring of Hydraulic Systems*. UCI Machine Learning Repository. https://doi.org/10.24432/C5CW21
